# Activation-Distribution-Aware LoRA Merging: Stage E Authentic GPU Training

**Project:** Activation-Distribution-Aware LoRA Merging  
**Target Base Model:** `Qwen/Qwen2.5-0.5B-Instruct`  
**Benchmark Tasks:** BoolQ (`google/boolq`), GSM8K (`openai/gsm8k`, `main`), ARC-Easy (`allenai/ai2_arc`, `ARC-Easy`)  
**Target Commit SHA:** `c7c6d62d3f8ebf547b71028327c9f7674960548a`  

---

### Strict Research Integrity Standards
1. **Authentic Datasets:** Real Hugging Face benchmarks only. Synthetic fallback is strictly forbidden in final mode.
2. **Benchmark Split Preservation:** Official evaluation partitions (BoolQ: 3,270, GSM8K: 1,319, ARC-Easy: 2,376) are 100% held-out and untouched.
3. **Calibration & Validation Isolation:** 128 calibration examples per task are reserved exclusively from training sources and never used for gradient updates.
4. **Exact Hyperparameter Parity:** $r=8, \alpha=16$, dropout=0.05, `q_proj`, `v_proj`, lr=$2\times 10^{-4}$, 3 epochs, batch size 4, grad accum 4.
5. **Unified Pipeline with Immediate Persistence:** Automated sequential execution via `scripts/train_all_adapters.py`, performing per-adapter validation, immediate Drive backup, and joint cross-task verification.

## 1. Hardware & Environment Diagnostics (CUDA GPU Check)

Colab must be configured with a GPU runtime (Runtime > Change runtime type > T4, V100, or A100 GPU).
This cell checks GPU availability, VRAM, RAM, disk space, and aborts immediately if CUDA is absent.

> **VRAM Note:** The 6 GB VRAM threshold is an initial lower-bound preflight heuristic. Real peak memory depends on dynamic token lengths, activation storage, and precision. A standard Colab T4 (15-16 GB) or A100 is recommended.

In [ ]:
import sys
import shutil
import platform
import torch

print("=" * 70)
print(" HARDWARE & RUNTIME DIAGNOSTICS ")
print("=" * 70)

print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")

# Check CUDA hardware
if not torch.cuda.is_available():
    raise SystemError(
        "CRITICAL ERROR: No CUDA GPU detected in this Colab session!\n"
        "Please go to: Runtime -> Change runtime type -> Hardware accelerator -> Select T4, V100, or A100 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU Model            : {gpu_name}")
print(f"GPU Total VRAM       : {total_vram_gb:.2f} GB")
print(f"CUDA Capability      : {torch.cuda.get_device_capability(0)}")

# Check RAM and Disk
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"System RAM           : {ram.available / (1024 ** 3):.2f} GB available / {ram.total / (1024 ** 3):.2f} GB total")
except ImportError:
    pass

total_d, used_d, free_d = shutil.disk_usage(".")
print(f"Disk Space           : {free_d / (1024 ** 3):.2f} GB free / {total_d / (1024 ** 3):.2f} GB total")
print("=" * 70)
print("Hardware Check: PASSED. CUDA GPU is active and ready for training.")

## 2. Mount Google Drive for Immediate Persistent Artifact Storage

Colab local disk `/content` is ephemeral and erased when sessions disconnect.
Mounting Google Drive ensures trained adapter checkpoints (`safetensors`), logs, and manifests persist permanently.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    persistent_base = Path('/content/drive/MyDrive/activation_aware_lora_merging_artifacts')
    persistent_base.mkdir(parents=True, exist_ok=True)
    (persistent_base / "adapters").mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted successfully. Persistent storage: {persistent_base}")
except Exception as e:
    print(f"Note: Running outside standard Colab drive mount ({e}). Artifacts will be saved locally.")
    persistent_base = Path("artifacts")
    persistent_base.mkdir(parents=True, exist_ok=True)

## 3. Install Repository Dependencies (Without Breaking CUDA PyTorch)

We do NOT reinstall PyTorch since Colab comes with a preconfigured CUDA-enabled PyTorch build.
We install only the necessary packages conforming to the repository's `requirements.txt`.

In [ ]:
# Install compatible ecosystem dependencies
!pip install -q transformers peft datasets accelerate safetensors pyyaml pytest scipy psutil

## 4. Repository Detection & Working Directory Setup

Detects if repository files are present, or clones from your repository.

In [ ]:
import os
import subprocess
from pathlib import Path

# Check working directory locations
if os.path.exists("configs/training.yaml"):
    print("Found repository in current working directory.")
elif os.path.exists("activation-aware-lora-merging/configs/training.yaml"):
    os.chdir("activation-aware-lora-merging")
    print("Navigated into 'activation-aware-lora-merging'.")
elif (persistent_base / "activation-aware-lora-merging/configs/training.yaml").exists():
    os.chdir(str(persistent_base / "activation-aware-lora-merging"))
    print(f"Navigated into persistent repo at: {persistent_base / 'activation-aware-lora-merging'}")
else:
    repo_url = os.environ.get("REPO_URL", "").strip()
    if not repo_url:
        repo_url = input("Enter your Git repository URL (or press Enter if uploading folder manually): ").strip()
    if repo_url:
        !git clone {repo_url} activation-aware-lora-merging
        %cd activation-aware-lora-merging

assert os.path.exists("configs/training.yaml"), (
    "Error: Repository root not found! Please ensure 'configs/training.yaml' exists in current directory."
)

try:
    commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
    print(f"Current Git Commit SHA: {commit_hash}")
    !git log -1 --oneline
except Exception:
    print("Git metadata not attached (standalone directory execution).")

## 5. Authentic Dataset Provenance & Benchmark Split Audit

Execute `scripts/audit_datasets.py` with `configs/experiment.yaml` in **FINAL** mode.
This confirms:
1. Pinned Hugging Face revisions loaded:
   - `google/boolq`: `35b264d03638db9f4ce671b711558bf7ff0f80d5`
   - `openai/gsm8k`: `740312add88f781978c0658806c59bc2815b9866`
   - `allenai/ai2_arc`: `210d026faf9955653af8916fad021475a3f00453`
2. Official evaluation partitions (3,270 / 1,319 / 2,376) 100% held out.
3. Upstream ARC-Easy duplicates excluded from training candidates.
4. Zero data leakage across splits.

In [ ]:
!python scripts/audit_datasets.py --config configs/experiment.yaml

## 6. Preflight Training & Configuration Parity Check

Run `scripts/train_all_adapters.py --preflight` to verify model architecture and hyperparameter parity.

In [ ]:
!python scripts/train_all_adapters.py --preflight

## 7. Run Unit Test Suite (50/50 Tests)

Confirm that all matrix utilities, SVD invariants, KnOTS algorithms, and benchmark policies pass.

In [ ]:
!pytest tests -v

## 8. Execute Unified Multi-Adapter GPU Training Pipeline (Primary Route)

This single command executes the complete multi-adapter training suite across BoolQ, GSM8K, and ARC-Easy on CUDA:
1. **Clean Directory Reset:** If any previous interrupted run existed, it is cleanly reset to prevent corrupting or mixing checkpoints.
2. **Task Training:** Trains BoolQ (9,199 train), GSM8K (7,245 train), and ARC-Easy (2,121 train) with exact hyperparameters ($r=8, \alpha=16, lr=2\times 10^{-4}, 3\text{ epochs}, b=4, \text{accum}=4$).
3. **Immediate Per-Task Verification:** Audits each adapter's tensors, finite weights, and PEFT validity immediately upon completion.
4. **Immediate Per-Task Google Drive Backup:** Mirrors the verified checkpoint to `/content/drive/MyDrive/.../adapters/<task>/` immediately.
5. **Joint Verification & Cross-Task Divergence:** Audits all three adapters together, confirming pairwise divergence, distinct SHA-256 checksums, and forward logits.
6. **Report Persistence:** Automatically backs up verification reports to Google Drive.

In [ ]:
!python scripts/train_all_adapters.py \
    --config configs/training.yaml \
    --experiment-config configs/experiment.yaml \
    --device cuda \
    --backup-dir /content/drive/MyDrive/activation_aware_lora_merging_artifacts

## 9. Final Independent Verification of Persistent Checkpoints

Confirms that all 3 adapters in the persistent backup directory pass verification independently.

In [ ]:
!python scripts/verify_adapters.py \
    --adapters-dir /content/drive/MyDrive/activation_aware_lora_merging_artifacts/adapters \
    --output-report /content/drive/MyDrive/activation_aware_lora_merging_artifacts/experiment/final_backup_verification.json \
    --device cuda

print("=" * 70)
print("STAGE E GPU TRAINING & VERIFICATION AUDIT COMPLETE.")
print("=" * 70)